# Module D - Low-Confidence Warning Testing on Google Colab

This notebook tests the **Low-Confidence Warning Feature** for Module D.

## What is Low-Confidence Warning?
When a search query returns poor results (confidence score < 0.20), the system displays:
```
⚠️ WARNING: Retrieved results may not be relevant.
⚠️ Matching confidence is low (score: 0.15).
⚠️ Consider rephrasing your query or checking translation quality.
```

## What We'll Test:
1. ✅ Good queries → NO warning (high confidence)
2. ⚠️ Bad queries → WARNING displayed (low confidence)
3. 🎯 Custom thresholds → Adjustable sensitivity
4. 📊 Score analysis → Understand confidence levels

## Prerequisites:
- Push your `clir-ly` project to GitHub
- Your `data/processed/articles_all.jsonl` file should be in the repo

Let's get started! 👇

## Step 1: Clone Repository from GitHub

**Before running**: Make sure your code is pushed to GitHub!

No Google Drive needed - we'll clone directly from your repo.

In [ ]:
# Clone your GitHub repository
# UPDATE THIS URL to your actual GitHub repo! 👇

GITHUB_REPO_URL = "https://github.com/AbDhrubo/clir-ly.git"  # 👈 UPDATE THIS

print(f"🔄 Cloning repository from: {GITHUB_REPO_URL}")
!git clone {GITHUB_REPO_URL}

print("\n✅ Repository cloned successfully!")

In [ ]:
# Change to the cloned repository directory
import os

# Extract repo name from the URL (usually "clir-ly")
REPO_NAME = "clir-ly"  # 👈 UPDATE if your repo has a different name

os.chdir(REPO_NAME)
print(f"✅ Changed to: {os.getcwd()}")
print(f"✅ Ready to install dependencies!")

In [ ]:
# Install dependencies
!pip install -q rank-bm25 sentence-transformers thefuzz rapidfuzz langdetect transformers torch

print("✅ Dependencies installed!")

## Step 2: Load Articles Dataset

We'll load 100 articles for quick testing (adjust `LIMIT` for more/less)

In [ ]:
import json
from src.retrieval.hybrid import HybridSearch

# Load articles
print("📂 Loading articles...")
LIMIT = 100  # Adjust this number (100 = fast, 1000 = more comprehensive)

articles = []
with open('data/processed/articles_all.jsonl', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= LIMIT:
            break
        articles.append(json.loads(line))

print(f"✅ Loaded {len(articles)} articles")
print(f"\nSample article titles:")
for i in range(min(3, len(articles))):
    print(f"  {i+1}. {articles[i].get('title', 'N/A')[:60]}")

## Step 3: Initialize Hybrid Search

This will load all three search methods (BM25, Fuzzy, Semantic).

**Note**: The semantic model download may take 1-2 minutes on first run.

In [ ]:
# Initialize Hybrid Search
print("🔧 Initializing Hybrid Search...")
print("   (This may take 1-2 minutes to download semantic model)\n")

hybrid = HybridSearch(articles)

print("\n✅ Hybrid Search is ready!")

## Test Case 1: Good Query (NO Warning Expected)

Let's search for a relevant query that should find good matches.

In [ ]:
print("="*80)
print("TEST CASE 1: Good Query → NO WARNING EXPECTED")
print("="*80)
print("Query: 'Bangladesh cricket team'\n")

results = hybrid.search("Bangladesh cricket team", k=5, verbose=False)

print(f"\n📊 Results:")
print(f"   Top result score: {results[0][1]:.3f}")
print(f"   Expected: Score > 0.20 (no warning should appear above)\n")

print("   Top 3 Results:")
for i, (doc_id, score, doc, breakdown) in enumerate(results[:3], 1):
    print(f"   {i}. Score: {score:.3f} | {doc.get('title', 'N/A')[:60]}")

## Test Case 2: Gibberish Query (WARNING Expected)

Let's search for complete nonsense that won't match anything.

In [ ]:
print("="*80)
print("TEST CASE 2: Gibberish Query → WARNING EXPECTED")
print("="*80)
print("Query: 'xyzqwerty asdfzxcv blahblah random nonsense'\n")

results = hybrid.search("xyzqwerty asdfzxcv blahblah random nonsense", k=5, verbose=False)

print(f"\n📊 Results:")
print(f"   Top result score: {results[0][1]:.3f}")
print(f"   Expected: Score < 0.20 (warning should appear above)")
print(f"   Status: {'✅ Working!' if results[0][1] < 0.20 else '⚠️ Score higher than expected'}")

## Test Case 3: Completely Unrelated Query (WARNING Expected)

Let's search for something totally unrelated to news articles.

In [ ]:
print("="*80)
print("TEST CASE 3: Unrelated Query → WARNING EXPECTED")
print("="*80)
print("Query: 'quantum mechanics photosynthesis algorithm'\n")

results = hybrid.search("quantum mechanics photosynthesis algorithm", k=5, verbose=False)

print(f"\n📊 Results:")
print(f"   Top result score: {results[0][1]:.3f}")
print(f"   Expected: Score < 0.20 (warning should appear above)")
print(f"   Status: {'✅ Working!' if results[0][1] < 0.20 else '⚠️ Score higher than expected'}")

## Test Case 4: Custom Threshold

You can adjust the warning threshold. Let's test with a stricter threshold (0.50).

In [ ]:
print("="*80)
print("TEST CASE 4: Custom Threshold (0.50) → Stricter Warning")
print("="*80)
print("Query: 'Bangladesh'\n")

results = hybrid.search("Bangladesh", k=5, verbose=False, confidence_threshold=0.50)

print(f"\n📊 Results:")
print(f"   Top result score: {results[0][1]:.3f}")
print(f"   Threshold: 0.50 (stricter than default 0.20)")
print(f"   Warning shown: {'Yes' if results[0][1] < 0.50 else 'No'}")

## Test Case 5: Cross-Lingual Query (Good Confidence Expected)

Let's test with a Bangla query to see cross-lingual performance.

In [ ]:
print("="*80)
print("TEST CASE 5: Bangla Query → Should Find Matches")
print("="*80)
print("Query: 'বাংলাদেশ ক্রিকেট' (Bangladesh cricket)\n")

results = hybrid.search("বাংলাদেশ ক্রিকেট", k=5, verbose=False)

print(f"\n📊 Results:")
print(f"   Top result score: {results[0][1]:.3f}")
print(f"   Expected: High confidence (semantic search handles cross-lingual)")

print("\n   Top 3 Results:")
for i, (doc_id, score, doc, breakdown) in enumerate(results[:3], 1):
    print(f"   {i}. Score: {score:.3f} | Lang: {doc.get('language', 'N/A')} | {doc.get('title', 'N/A')[:50]}")

## Score Analysis & Visualization

Let's analyze confidence scores across multiple queries to understand the distribution.

In [ ]:
# Test multiple queries and analyze score distribution
test_queries = [
    ("Bangladesh cricket team", "good"),
    ("ঢাকা শহর", "good"),
    ("politics government election", "good"),
    ("random gibberish xyz", "bad"),
    ("quantum physics aliens", "bad"),
    ("zzzz qqqqq wwwww", "bad"),
]

print("="*80)
print("SCORE ANALYSIS ACROSS MULTIPLE QUERIES")
print("="*80)

results_summary = []

for query, expected_quality in test_queries:
    results = hybrid.search(query, k=5, verbose=False)
    top_score = results[0][1]
    
    results_summary.append({
        'query': query[:40],
        'expected': expected_quality,
        'score': top_score,
        'warning': top_score < 0.20
    })

# Display results
print(f"\n{'Query':<42} | {'Expected':<6} | {'Score':<6} | {'Warning'}")
print("-" * 80)

for r in results_summary:
    warning_icon = "⚠️ YES" if r['warning'] else "✅ NO"
    print(f"{r['query']:<42} | {r['expected']:<6} | {r['score']:.3f}  | {warning_icon}")

# Summary
good_queries = [r for r in results_summary if r['expected'] == 'good']
bad_queries = [r for r in results_summary if r['expected'] == 'bad']

avg_good = sum(r['score'] for r in good_queries) / len(good_queries)
avg_bad = sum(r['score'] for r in bad_queries) / len(bad_queries)

print("\n" + "="*80)
print("📊 SUMMARY:")
print(f"   Average score for GOOD queries: {avg_good:.3f}")
print(f"   Average score for BAD queries:  {avg_bad:.3f}")
print(f"   Score difference: {avg_good - avg_bad:.3f}")
print("\n   ✅ Feature working correctly!" if avg_good > avg_bad else "   ⚠️ Check implementation")
print("="*80)

## Visualization (Optional)

Let's create a simple bar chart to visualize the confidence scores.

In [ ]:
import matplotlib.pyplot as plt

# Prepare data
queries = [r['query'][:20] + '...' if len(r['query']) > 20 else r['query'] for r in results_summary]
scores = [r['score'] for r in results_summary]
colors = ['green' if r['expected'] == 'good' else 'red' for r in results_summary]

# Create bar chart
plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(queries)), scores, color=colors, alpha=0.6)

# Add threshold line
plt.axhline(y=0.20, color='orange', linestyle='--', linewidth=2, label='Warning Threshold (0.20)')

# Customize
plt.xlabel('Queries', fontsize=12)
plt.ylabel('Confidence Score', fontsize=12)
plt.title('Low-Confidence Warning Feature: Score Distribution', fontsize=14, fontweight='bold')
plt.xticks(range(len(queries)), queries, rotation=45, ha='right')
plt.ylim(0, 1.0)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

plt.show()

print("\n📊 Interpretation:")
print("   🟢 Green bars = Good queries (should be above threshold)")
print("   🔴 Red bars = Bad queries (should be below threshold)")
print("   🟠 Orange line = Warning threshold (0.20)")

## Test Your Own Queries

Now you can test your own queries interactively!

In [ ]:
# Interactive testing
print("="*80)
print("INTERACTIVE QUERY TESTING")
print("="*80)
print("Enter your queries below (or modify the query variable)\n")

# Change this to test different queries
your_query = "Dhaka traffic jam"  # 👈 CHANGE THIS

print(f"Query: '{your_query}'\n")
results = hybrid.search(your_query, k=5, verbose=False)

print(f"\n📊 Top 5 Results:")
for i, (doc_id, score, doc, breakdown) in enumerate(results, 1):
    print(f"\n{i}. Score: {score:.3f}")
    print(f"   Title: {doc.get('title', 'N/A')}")
    print(f"   Language: {doc.get('language', 'N/A')}")
    print(f"   Score Breakdown:")
    print(f"      BM25: {breakdown['bm25']:.3f}")
    print(f"      Fuzzy: {breakdown['fuzzy']:.3f}")
    print(f"      Semantic: {breakdown['semantic']:.3f}")

## ✅ Verification Checklist

After running all cells above, verify:

- [x] **Test 1** (Good query): Score > 0.20, NO warning appeared
- [x] **Test 2** (Gibberish): Score < 0.20, WARNING appeared
- [x] **Test 3** (Unrelated): Score < 0.20, WARNING appeared
- [x] **Test 4** (Custom threshold): Warning appeared if score < 0.50
- [x] **Test 5** (Cross-lingual): Decent score, semantic search working
- [x] **Score Analysis**: Good queries score higher than bad queries
- [x] **Visualization**: Chart shows clear separation between good/bad queries

## 🎉 Conclusion

If all tests passed:
- ✅ Low-confidence warning feature is working correctly!
- ✅ The system can distinguish between good and bad queries
- ✅ Cross-lingual search with confidence scoring is functional

## Next Steps (Module D Completion):
1. ✅ Low-confidence warning - DONE!
2. ⏳ Query execution time breakdown
3. ⏳ Comparison with classical search engines
4. ⏳ Full evaluation with labeled queries
5. ⏳ Detailed error analysis (5 categories)